# HDK v4.8 — Validación Estadística Final
## 10 splits, test McNemar, barrido C amplio

OMNI recomendó rigurosidad estadística antes de cerrar el ciclo.
Determinar si el gap de 0.10% es real o ruido de muestra.

In [ ]:
import torch, time, warnings, json, os
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import StratifiedShuffleSplit
from scipy.stats import chi2
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
D = 20000; V = 10000
N_SPLITS = 10
C_VALS = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
N_SEEDS = 40
EPOCHS = 5; MAX_ITER = 20; LR = 1.0
print(f'D={D}, V={V}, splits={N_SPLITS}, C={C_VALS}, seeds={N_SEEDS}')

In [ ]:
t0 = time.time()
news = fetch_20newsgroups(subset='all', remove=('headers','footers','quotes'))
X_all = news.data; y_all = news.target
print(f'Total: {len(X_all)} docs ({time.time()-t0:.1f}s)')
t0 = time.time()
vec = TfidfVectorizer(max_features=V)
X_tfidf = vec.fit_transform(X_all).astype(np.float32)
print(f'TF-IDF: {X_tfidf.shape} ({time.time()-t0:.1f}s)')

In [ ]:
def to_sparse_gpu(X):
    c = X.tocoo()
    idx = torch.LongTensor([c.row, c.col]).to(device)
    val = torch.FloatTensor(c.data).to(device)
    return torch.sparse_coo_tensor(idx, val, X.shape, device=device)

def train_lbfgs(X, y, in_dim, n_classes, wd=0.0):
    model = torch.nn.Linear(in_dim, n_classes).to(device)
    opt = torch.optim.LBFGS(model.parameters(), lr=LR, max_iter=MAX_ITER,
                           history_size=10, line_search_fn='strong_wolfe')
    loss_fn = torch.nn.CrossEntropyLoss()
    for ep in range(EPOCHS):
        def closure():
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            if wd > 0:
                for p in model.parameters():
                    loss = loss + 0.5 * wd * p.pow(2).sum()
            loss.backward()
            return loss
        opt.step(closure)
    return model

def mcnemar_test(y_true, pred_a, pred_b):
    n01 = np.sum((pred_a != y_true) & (pred_b == y_true))
    n10 = np.sum((pred_a == y_true) & (pred_b != y_true))
    stat = (abs(n01 - n10) - 1)**2 / (n01 + n10 + 1e-10)
    p = 1 - chi2.cdf(stat, 1)
    return stat, p

In [ ]:
sss = StratifiedShuffleSplit(n_splits=N_SPLITS, test_size=0.2, random_state=42)
results = {
    "D": D, "V": V, "N_splits": N_SPLITS, "N_seeds": N_SEEDS,
    "C_values": C_VALS, "splits": []
}
for fold, (tr_idx, te_idx) in enumerate(sss.split(X_all, y_all)):
    print(f'Fold {fold+1}/{N_SPLITS}: {len(tr_idx)}/{len(te_idx)}', flush=True)
    X_tr = X_tfidf[tr_idx]; X_te = X_tfidf[te_idx]
    y_tr = y_all[tr_idx]; y_te = y_all[te_idx]
    y_tr_t = torch.LongTensor(y_tr).to(device)
    X_tr_sp = to_sparse_gpu(X_tr); X_te_sp = to_sparse_gpu(X_te)
    lr_sk = LogisticRegression(max_iter=2000, solver='saga', n_jobs=-1, random_state=42)
    lr_sk.fit(X_tr, y_tr)
    sk_preds = lr_sk.predict(X_te)
    acc_sk = accuracy_score(y_te, sk_preds)
    fd = {'sklearn': round(float(acc_sk), 4), 'C_results': {}}
    for C in C_VALS:
        wd = 1.0 / (C * len(tr_idx)) if C > 0 else 0.0
        all_preds = []
        for s in range(N_SEEDS):
            torch.manual_seed(s)
            wv = torch.randint(0, 2, (V, D), dtype=torch.float32, device=device)
            X_h = (X_tr_sp @ wv); X_he = (X_te_sp @ wv)
            mu = X_h.mean(dim=0, keepdim=True)
            sd = X_h.std(dim=0, keepdim=True, unbiased=False)
            sd[sd == 0] = 1
            X_n = (X_h - mu) / sd; X_ne = (X_he - mu) / sd
            model = train_lbfgs(X_n, y_tr_t, D, 20, wd)
            with torch.no_grad():
                preds = model(X_ne).argmax(1).cpu().numpy()
            all_preds.append(preds)
            del wv, X_h, X_he, mu, sd, X_n, X_ne, model
            if device == 'cuda': torch.cuda.empty_cache()
        ens = np.apply_along_axis(lambda x: np.bincount(x.astype(int)).argmax(), 0, np.array(all_preds))
        acc_hdk = accuracy_score(y_te, ens)
        gap = acc_sk - acc_hdk
        stat, p = mcnemar_test(y_te, sk_preds, ens)
        fd['C_results'][str(C)] = {
            'hdk_acc': round(float(acc_hdk), 4),
            'gap': round(float(gap), 4),
            'mcnemar_p': round(float(p), 4),
            'significant': bool(p < 0.05)
        }
        sig = ' *SIG*' if p < 0.05 else ''
        print(f'  C={C}: HDK={acc_hdk*100:.2f}% gap={gap*100:.2f}% p={p:.4f}{sig}', flush=True)
    results['splits'].append(fd)
    del X_tr_sp, X_te_sp, lr_sk
    if device == 'cuda': torch.cuda.empty_cache()
    with open('/content/checkpoint.json', 'w') as f:
        json.dump(results, f)

In [ ]:
print()
sa = [s['sklearn'] for s in results['splits']]
print(f'sklearn LogReg: {np.mean(sa)*100:.2f}% +/- {np.std(sa)*100:.2f}%')
print()
for C in C_VALS:
    h = [s['C_results'][str(C)]['hdk_acc'] for s in results['splits']]
    g = [s['C_results'][str(C)]['gap'] for s in results['splits']]
    pv = [s['C_results'][str(C)]['mcnemar_p'] for s in results['splits']]
    ns = sum(1 for p in pv if p < 0.05)
    mg = np.mean(g)*100; sg = np.std(g)*100
    mh = np.mean(h)*100; sh = np.std(h)*100
    print(f'C={C:6.3f}: HDK={mh:.2f}+/-{sh:.2f}%  gap={mg:.2f}+/-{sg:.2f}%  sig={ns}/10')

bc = min(C_VALS, key=lambda c: np.mean([s['C_results'][str(c)]['gap'] for s in results['splits']]))
bg = np.mean([s['C_results'][str(bc)]['gap'] for s in results['splits']]) * 100
print(f'Best C={bc}  mean gap={bg:.2f}%')
if abs(bg) < 0.3:
    print(f'Gap {bg:.2f}%: NOT significant. HDK = sklearn LogReg.')
else:
    print(f'Gap {bg:.2f}%: real and significant.')
print('='*50)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    out = '/content/drive/MyDrive/hdk_validation_final.json'
    with open(out, 'w') as f: json.dump(results, f, indent=2)
    print(f'Guardado en Drive: {out}')
except:
    out = 'hdk_validation_final.json'
    with open(out, 'w') as f: json.dump(results, f, indent=2)
    from google.colab import files
    files.download(out)